# Extract Measures — Response-Locked

Extracts peak and mean amplitude measures from response-locked evokeds.

- **Flow (Castellar)**: positive peak [-100, 0ms] + negative peak [0, 300ms] or [160, 240ms], peak-to-peak amplitude and latency
- **Load (P3)**: mean amplitude per window on all channels

CSD and baseline correction are set automatically per paper:
- Flow: CSD=True, bc-200-0ms
- Load: CSD=False, bc-200-100ms

**Usage**: set `PAPER` in Cell 1, then run all.

Last edit: 15.10.2026 @anguyen

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from tqdm import tqdm

def find_repo_root(marker='core') -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not locate repo root (no '{marker}/' found above {Path.cwd()})")

sys.path.insert(0, str(find_repo_root() / 'core'))
import config_flow, config_load
from participants import get_participants
from utils import load_pkl, show_latest_pkls

# ==============================================================================
# CONFIGURATION — edit here
# ==============================================================================

PAPER: str = 'flow'    # 'flow' | 'load'

# ==============================================================================
# FIXED PER PAPER — do not change
# ==============================================================================

PARADIGM = 'active'   # response-locked is always active

if PAPER == 'flow':
    CSD           = True
    BASELINE_CORR = 'bc-200-0'
elif PAPER == 'load':
    CSD           = False
    BASELINE_CORR = 'bc-200-100'

label_csd = 'CSD' if CSD else 'NOCSD'
DATE_STR  = datetime.today().strftime('%Y%m%d')

if sys.platform == 'win32':
    root = Path('L:/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2')
else:
    root = Path('/Volumes/faculty/EEG_Flow_data/Study_EEG_Flow_Active/derivatives_v2')

DATE_SOURCE = show_latest_pkls(root, PAPER, PARADIGM, 'resp', 'raw', BASELINE_CORR, None)
print(f'Paper: {PAPER.upper()} | Baseline: {BASELINE_CORR} | CSD: {CSD}')

In [ ]:
# ==============================================================================
# Derived labels and paths (auto — do not edit)
# ==============================================================================

_cfg       = config_flow if PAPER == 'flow' else config_load
CONDITIONS = _cfg.CONDITIONS

n = len(get_participants(PAPER, PARADIGM))

input_path  = root / 'outputs' / 'grand_averages' / PAPER / 'raw'
output_path = root / 'outputs' / 'measures' / PAPER
output_path.mkdir(parents=True, exist_ok=True)

print(f'Input : {input_path}')
print(f'Output: {output_path}')

In [ ]:
# ==============================================================================
# Load pkl files
# ==============================================================================

def _build_fname(cond_key: str) -> str:
    base = f'{DATE_SOURCE}_{PARADIGM}_{n}p_cond-{cond_key}'
    return f'{base}_resp_{BASELINE_CORR}_{label_csd}_raw.pkl'


evokeds: dict[str, dict] = {}
for plot_label, cond_key in CONDITIONS.items():
    fname = _build_fname(cond_key)
    evokeds[plot_label] = load_pkl(input_path / fname)
    print(f'Loaded [{plot_label}] <- {fname}')
    print(f'  keys: {list(evokeds[plot_label].keys())}')

In [ ]:
# ==============================================================================
# Helper functions
# ==============================================================================

def get_cluster_mean(evoked, channels):
    """Average evoked data across a cluster of channels, return 1D array (times,)."""
    picks = [evoked.ch_names.index(ch) for ch in channels if ch in evoked.ch_names]
    if not picks:
        raise ValueError(f'None of {channels} found in evoked')
    return evoked.data[picks, :].mean(axis=0)


def find_peak(data, times_ms, tmin_ms, tmax_ms, mode='neg'):
    """Find peak amplitude and latency within a time window.

    Parameters
    ----------
    data     : 1D array (times,) in Volts
    times_ms : 1D array of times in ms
    tmin_ms  : window start in ms
    tmax_ms  : window end in ms
    mode     : 'neg' for most negative, 'pos' for most positive

    Returns
    -------
    amp_uv : float, peak amplitude in uV
    lat_ms : float, peak latency in ms
    """
    mask = (times_ms >= tmin_ms) & (times_ms <= tmax_ms)
    if not np.any(mask):
        return np.nan, np.nan
    data_win  = data[mask] * 1e6
    times_win = times_ms[mask]
    idx = np.argmin(data_win) if mode == 'neg' else np.argmax(data_win)
    return float(data_win[idx]), float(times_win[idx])


def mean_around_peak(data, times_ms, peak_lat_ms, half_window_ms=25):
    """Mean amplitude in uV in a window around the peak latency."""
    mask = (
        (times_ms >= peak_lat_ms - half_window_ms) &
        (times_ms <= peak_lat_ms + half_window_ms)
    )
    if not np.any(mask):
        return np.nan
    return float(data[mask].mean() * 1e6)

print('Helper functions ready.')

## Flow — Castellar extraction

Positive peak [-100, 0ms] + negative peak in [0, 300ms] or [160, 240ms].  
Three configurations: FCz only, frontal cluster × 0-300ms, frontal cluster × 160-240ms.

In [ ]:
# ==============================================================================
# Flow Castellar — peak-to-peak extraction (following Castellar et al.)
# pre-response:  most positive in [-100, 0ms]
# post-response: most negative in [0, 300ms] (or [160, 240ms])
# p2p_amp = pre_peak - post_peak  (positive value)
# p2p_lat = post_lat - pre_lat    (positive value)
# ==============================================================================

if PAPER != 'flow':
    print("Skipping Castellar extraction — set PAPER='flow' to run.")
else:
    CONFIGS = [
        {'label': 'FCz_0-300',
         'channels': ['FCz'],
         'post_window': (0, 300)},
        {'label': 'frontal_0-300',
         'channels': ['F1', 'Fz', 'F2', 'FC1', 'FCz', 'FC2'],
         'post_window': (0, 300)},
        {'label': 'frontal_160-240',
         'channels': ['F1', 'Fz', 'F2', 'FC1', 'FCz', 'FC2'],
         'post_window': (160, 240)},
    ]
    PRE_WINDOW = (-100, 0)

    rows = []
    for plot_label, cond_key in tqdm(CONDITIONS.items(), desc='Conditions'):
        key = f'{cond_key}_response'
        if key not in evokeds[plot_label] or not evokeds[plot_label][key]:
            print(f"Warning: '{key}' not found or empty — skipping.")
            continue

        for evoked in evokeds[plot_label][key]:
            participant = evoked.comment if evoked.comment else 'unknown'
            times_ms    = evoked.times * 1000
            ch_types   = set(evoked.get_channel_types())
            pick_type  = 'csd' if 'csd' in ch_types else 'eeg'
            evoked_eeg = evoked.copy().pick(pick_type)

            for cfg in CONFIGS:
                try:
                    data = get_cluster_mean(evoked_eeg, cfg['channels'])  # use evoked_eeg
                except ValueError as e:
                    print(f'  {participant} | {cfg["label"]}: {e}')
                    continue

                pre_amp,  pre_lat  = find_peak(data, times_ms, *PRE_WINDOW,         mode='pos')
                post_amp, post_lat = find_peak(data, times_ms, *cfg['post_window'],  mode='neg')

                p2p_amp = pre_amp - post_amp if not np.isnan(pre_amp + post_amp) else np.nan
                p2p_lat = post_lat - pre_lat if not np.isnan(pre_lat + post_lat) else np.nan

                rows.append({
                    'participant':      participant,
                    'condition':        plot_label,
                    'channel_config':   cfg['label'],
                    'pre_window':       f'{PRE_WINDOW[0]}-{PRE_WINDOW[1]}ms',
                    'post_window':      f"{cfg['post_window'][0]}-{cfg['post_window'][1]}ms",
                    'pre_peak_amp_uv':  pre_amp,
                    'pre_peak_lat_ms':  pre_lat,
                    'post_peak_amp_uv': post_amp,
                    'post_peak_lat_ms': post_lat,
                    'p2p_amp_uv':       p2p_amp,
                    'p2p_lat_ms':       p2p_lat,
                })

    df_castellar = pd.DataFrame(rows)
    n_castellar  = df_castellar['participant'].nunique()
    print(f'\nCastellar: {len(df_castellar):,} rows | {n_castellar} participants')
    print(df_castellar.head())

    fname = output_path / f'{DATE_STR}_{PARADIGM}_{n_castellar}p_flow_resp_{BASELINE_CORR}_{label_csd}_castellar_p2p.csv'
    df_castellar.to_csv(fname, index=False)
    print(f'Saved -> {fname.name}')

## Load — P3 extraction

Mean amplitude per window on all channels, three window definitions tested.

In [ ]:
# ==============================================================================
# Load P3 — mean amplitude per window
# ==============================================================================

if PAPER != 'load':
    print("Skipping Load P3 extraction — set PAPER='load' to run.")
else:
    P3_WINDOWS = [
        (-50,  100),
        (-100, 100),
        (-75,  120),
    ]

    rows_p3 = []

    for window in P3_WINDOWS:
        win_str = f'{window[0]}-{window[1]}ms'

        for plot_label, cond_key in tqdm(CONDITIONS.items(), desc=f'Conditions {win_str}'):
            key = f'{cond_key}_response'
            if key not in evokeds[plot_label]:
                continue

            for evoked in evokeds[plot_label][key]:
                participant = evoked.comment or 'unknown'
                times_ms    = evoked.times * 1000
                mask        = (times_ms >= window[0]) & (times_ms <= window[1])

                # pick EEG only to avoid mixed units from misc channels
                evoked_eeg = evoked.copy().pick('eeg')

                for ch_name in evoked_eeg.ch_names:
                    ch_idx = evoked_eeg.ch_names.index(ch_name)
                    amp    = float(evoked_eeg.data[ch_idx, mask].mean() * 1e6)
                    rows_p3.append({
                        'participant': participant,
                        'condition':   plot_label,
                        'ch_name':     ch_name,
                        'window':      win_str,
                        'mean_amp_uv': amp,
                    })

    df_p3 = pd.DataFrame(rows_p3)
    n_p3  = df_p3['participant'].nunique()
    fname = output_path / f'{DATE_STR}_{PARADIGM}_{n_p3}p_load_resp_{BASELINE_CORR}_{label_csd}_p3.csv'
    df_p3.to_csv(fname, index=False)
    print(f'Saved -> {fname.name} | {len(df_p3)} rows | windows: {df_p3["window"].unique()}')

## Sanity check

In [ ]:
if PAPER == 'flow' and 'df_castellar' in dir():
    print('=== CASTELLAR ===')
    print(f'Rows         : {len(df_castellar):,}')
    print(f'Participants : {df_castellar["participant"].nunique()}')
    print(f'Conditions   : {sorted(df_castellar["condition"].unique())}')
    print(f'Configs      : {sorted(df_castellar["channel_config"].unique())}')
    print(f'Any NaN p2p  : {df_castellar["p2p_amp_uv"].isna().sum()} rows')
    print('\nMean p2p amplitude per condition and config:')
    print(df_castellar.groupby(['condition', 'channel_config'])['p2p_amp_uv'].mean().round(2).to_string())

elif PAPER == 'load' and 'df_p3' in dir():
    print('=== P3 ===')
    print(f'Rows         : {len(df_p3):,}')
    print(f'Participants : {df_p3["participant"].nunique()}')
    print(f'Conditions   : {sorted(df_p3["condition"].unique())}')
    print(f'Windows      : {sorted(df_p3["window"].unique())}')
    print(f'Channels     : {df_p3["ch_name"].nunique()}')
    print(f'Any NaN      : {df_p3["mean_amp_uv"].isna().sum()} rows')
    print('\nMean amplitude per condition and window (parietal cluster):')
    PARIETAL = ['P1', 'Pz', 'P2', 'POz']
    print(
        df_p3[df_p3['ch_name'].isin(PARIETAL)]
        .groupby(['condition', 'window'])['mean_amp_uv']
        .mean().round(2).to_string()
    )